# 1) Setup



### a) Standard imports

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import regex as re
import sys
from pathlib import Path
import os

# Standard imports for data processing and visualization
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px
import gensim

print("Imports successful")

Imports successful


### b) Custom modules

In [ ]:
# Import custom modules
# Add modules to path
sys.path.insert(0, '/project/ssd-stu-research/ploertscher/thesis_code/ideological_resonance_thesis/w2v/analysis/modules')

import pca_util
import importlib
import semaxis_util
import semanalysis_util
import helpers
importlib.reload(pca_util)
importlib.reload(semaxis_util)
importlib.reload(semanalysis_util)
importlib.reload(helpers)

from pca_util import create_actor_action_matrix
from semaxis_util import SemAxis, anch2vec, anch2conceptvec, axis_parallelism, pair_parallelism, find_antonym, find_antonyms_fullsearch
from semanalysis_util import (
    actor_embeddings_from_w2v_entities,
    actor_embd, actor_proj, compare_semantic_to_pca,
    association_matrix, compare_verb_loadings, visualize_projection,
)
from helpers import *


print("Imports successful")


Imports successful


### c) Word2Vec model

In [3]:
# Trained Word2Vec (saved under w2v/models/<MODEL_NAME>/ by train_w2v_cpu.py)
MODEL_NAME = "2_w2v_min10"

print("Loading trained Word2Vec...")
w2v_model = helpers.load_trained_w2v_keyed_vectors(MODEL_NAME)
# L2-normalize vectors in-place for semantic-axis / projection math
if hasattr(w2v_model, "init_sims"):
    w2v_model.init_sims(replace=True)
    print("Normalized Vectors to unit length")
else:
    print("Cannot normalize: init_sims missing")

print(f"Loaded run {MODEL_NAME!r}. Vocabulary size: {len(w2v_model):,}")
print(f"Vector size: {w2v_model.vector_size}")

# Optional: maps for PCA (canonical) ↔ w2v underscore tokens
token_to_canonical = helpers.load_w2v_token_to_canonical()
canonical_to_w2v = helpers.load_canonical_to_w2v_token()
print(f"Entity map: {len(token_to_canonical)} w2v tokens → canonical")


Loading trained Word2Vec...
Normalized Vectors to unit length
Loaded run '2_w2v_min10'. Vocabulary size: 177,584
Vector size: 300
Entity map: 132 w2v tokens → canonical


/scratch/local/jobs/49151400/ipykernel_1589104/2429661334.py:8: DeprecationWarning: Call to deprecated `init_sims` (Use fill_norms() instead. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)


Let's inspect the word2vec model for quality.

In [ ]:
# Define tokens for easy adjustment
similarity_token = "media"
analogy_tokens = {
    "positive": ["media", "social"],
    "negative": ["mainstream"],
}

# Find the 10 closest tokens to the selected token
most_similar_results = w2v_model.most_similar(similarity_token, topn=10)
print(f"Tokens closest to '{similarity_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

print("\nWord analogies:")
try:
    analogy_results = w2v_model.most_similar(
        positive=analogy_tokens["positive"],
        negative=analogy_tokens["negative"],
        topn=10
    )
    positive_str = " + ".join(analogy_tokens["positive"])
    negative_str = " - " + " - ".join(analogy_tokens["negative"]) if analogy_tokens["negative"] else ""
    print(f"{positive_str}{negative_str} is closest to:")
    for tok, sim_score in analogy_results:
        print(f"{tok:20s} {sim_score:.3f}")
except KeyError as e:
    print(f"Token not in vocabulary: {e}")

Tokens closest to 'media':
mainstream           0.696
msm                  0.599
social               0.586
medias               0.546
donaldtrumpjrofficialchannel 0.540
outlet               0.532
outlets              0.529
lamestream           0.527
robbinsville         0.513
propaganda           0.490

Word analogies:
media + social - mainstream is closest to:
bscsca               0.467
robbinsville         0.454
platforms            0.447
socail               0.426
leegarrettnews       0.424
medias               0.421
tokeninsigh          0.417
communcation         0.414
dextool              0.411
retargeting          0.408


# 2) Construct Semantic Axes

In this section, I will:
1. Define antonym pairs representing semantic dimensions
2. Create semantic axes using the `SemAxis` class
3. Evaluate axis quality using parallelism metrics
4. Refine axes by removing weak pairs or adding strong ones

## Epistemic Action Axis

In [15]:
# First, let's create an "epistemic_action" semantic axis using a set of predefined antonym pairs
# Each pair is (positive_pole, negative_pole)
epistemic_pairs = [
    ('hide', 'reveal')
]

# Create semantic axis using the SemAxis class defined in semaxis_util.py
epistemic_action_axis = SemAxis(
    epistemic_pairs, 
    w2v_model, 
    name="epistemic_action"
)

# View axis summary
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 1
Concept vector dimension: 300


In [16]:
# Find new pair to add
candidates = ['conceal', 'mask', 'veil']
best = epistemic_action_axis.find_best_antonym('expose', candidates)
print(best)
print(f"Best new pair: expose-{best[0][0]} ({best[0][1]:.3f})")

[('conceal', 0.2250729501247406), ('mask', 0.11307818442583084), ('veil', 0.07659771293401718)]
Best new pair: expose-conceal (0.225)


In [17]:
# Add pair
epistemic_action_axis = epistemic_action_axis.add_pair(('conceal', 'expose'))
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.225

Best pairs (highest parallelism):
  ('hide', 'reveal'): 0.225
  ('conceal', 'expose'): 0.225

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.225
  ('conceal', 'expose'): 0.225


In [ ]:
# Looking for top N best antonyms
f, r = epistemic_action_axis.find_antonyms_fullsearch("suppress", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

# There's a fair amount of noise in the antonyms, but "leak" seems to be the most sensible match fitting out epistemic action axis.

Word-Score: seanhannity - 0.3659227431283588
Word-Score: realrecently - 0.35154604002189505
Word-Score: 𝗛𝗲𝗴𝘀𝗲𝘁𝗵 - 0.3512827434711908
Word-Score: 30pmest - 0.3481608485954734
Word-Score: trumpmedia - 0.3414616311272804
Word-Score: exspose - 0.3405601169532514
Word-Score: annouced - 0.33887989954665115
Word-Score: 25st - 0.3352495707726353
Word-Score: trumpify - 0.3306112913683512
Word-Score: acсount - 0.3293545283225446
Word-Score: dobbs - 0.3288818999033069
Word-Score: 𝙋𝙖𝙩𝙧𝙞𝙤𝙩𝙨 - 0.3272113709944332
Word-Score: bhtv - 0.32626766559255393
Word-Score: oceanbluesques - 0.3261663738257232
Word-Score: kaylaigh - 0.32611861930072233
Word-Score: huckabee - 0.3250616903658291
Word-Score: 21in - 0.3214424352185779
Word-Score: blakcout - 0.3212187375083747
Word-Score: 𝗣𝗲𝘁𝗲𝗿 - 0.3211031733346383
Word-Score: ѕaid - 0.32058472975042573
Word-Score: ѕpeak - 0.3196276424049632
Word-Score: lahren - 0.3189502223443619
Word-Score: killstream - 0.3185711072259444
Word-Score: rauch - 0.31856469486628614
Wor

In [ ]:
# Add pair
epistemic_action_axis = epistemic_action_axis.add_pair(('suppress', 'leak'))
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.241

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.249
  ('conceal', 'expose'): 0.248
  ('hide', 'reveal'): 0.226

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.226
  ('conceal', 'expose'): 0.248
  ('suppress', 'leak'): 0.249


In [79]:
candidates = [("obfuscate", "illuminate"),
            ("bury", "surface"),
            ("cover_up", "uncover"),
            ("muffle", "announce"),
            ("censor", "publish")]

for good, bad in candidates:
    try:
        best = epistemic_action_axis.find_best_antonym(good, [bad])
        print(f"Pair parallelism to axis: {good}-{bad} ({best[0][1]:.3f})")
    except ValueError as err:
        print(err)

Pair parallelism to axis: obfuscate-illuminate (0.197)
Pair parallelism to axis: bury-surface (0.179)
Pair parallelism to axis: cover_up-uncover (0.169)
Pair parallelism to axis: muffle-announce (0.186)
Pair parallelism to axis: censor-publish (0.189)


In [13]:
# Looking for top N best antonyms
f, r = epistemic_action_axis.find_antonyms_fullsearch("enlighten", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: distort - 0.29369850008760473
Word-Score: discredit - 0.2834342449168824
Word-Score: deny - 0.27248858285823413
Word-Score: circumvent - 0.2714514421216633
Word-Score: mediacide - 0.26959622354429774
Word-Score: misdirect - 0.26573907804203567
Word-Score: misrepresent - 0.2628281019688056
Word-Score: delegitimise - 0.26273246449825893
Word-Score: disinform - 0.25608741525180295
Word-Score: distract - 0.25564388639882324
Word-Score: confuse - 0.25315462256793986
Word-Score: numerator - 0.25277424145097876
Word-Score: omitting - 0.2525943811592434
Word-Score: manipulate - 0.25118935476403037
Word-Score: thwart - 0.24984499726493886
Word-Score: ignore - 0.2485415413558889
Word-Score: dissuade - 0.24617056417434519
Word-Score: lawfull - 0.24602062208435266
Word-Score: dehumanise - 0.24575300092172092
Word-Score: mislead - 0.24524562523320848
Word-Score: dishonestly - 0.24498084618563637
Word-Score: deriliction - 0.24399649237126642
Word-Score: undermine - 0.24388781310129976
Wo

## Hidden: Epistemic-action conjugations

Below is the code used to experiment with adding/removing individual combinations from the conjugations. While each one may individually fit the epistemic axis fairly well, their interaction in the axis can lead to low overall parallelism scores. Ultimately, I kept only the `suppresses-leaks` pair; this ensured that all pairs in the axis kept an individual score above 0.2 (strong).

In [ ]:
# Checking conjugations
conj = [('suppressing', 'leaking'), ('suppresses', 'leaks'), ('concealing', 'exposing'), ('conceals', 'exposes'), ('hiding', 'revealing'), ('hides', 'reveals')]
for bad, good in conj:
    candidates = [bad]
    best = epistemic_action_axis.find_best_antonym(good, candidates)
    print(f"{good} - {best[0][0]} ({best[0][1]:.3f})")

leaking - suppressing (0.243)
leaks - suppresses (0.242)
exposing - concealing (0.208)
exposes - conceals (0.171)
revealing - hiding (0.157)
reveals - hides (0.198)


In [49]:
# Adding one pair at a time to check if interaction between pairs is significant
epistemic_action_axis = epistemic_action_axis.add_pair(('hides', 'reveals'))

# Calculate overall parallelism
summary = epistemic_action_axis.summary()
match = re.search(r"Overall parallelism:\s*([-\d\.]+)", summary)
parallelism_score = float(match.group(1))
print(f"Overall parallelism: {parallelism_score}")

# Rechecking conjugations
conj = [('hiding', 'revealing')]
for bad, good in conj:
    candidates = [bad]
    best = epistemic_action_axis.find_best_antonym(good, candidates)
    print(f"{good} - {best[0][0]} ({best[0][1]:.3f})")

# Removing weaker pairs
# epistemic_action_axis = epistemic_action_axis.remove_pair(('conceals', 'exposes'))

Overall parallelism: 0.205
revealing - hiding (0.173)


In [55]:
epistemic_action_axis = epistemic_action_axis.remove_pair(('hides', 'reveals'))
# epistemic_action_axis = epistemic_action_axis.add_pair(('conceals', 'exposes'))
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.242

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.315
  ('suppresses', 'leaks'): 0.244
  ('conceal', 'expose'): 0.207

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.203
  ('conceal', 'expose'): 0.207
  ('suppresses', 'leaks'): 0.244


In [ ]:

for pair in conj:
    if pair not in [('exposes', 'conceals'), ('revealing', 'hiding')]:
        epistemic_action_axis = epistemic_action_axis.add_pair(pair)
        print(epistemic_action_axis.summary())
        i += 1
        if i > 1:
            break
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.241

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.249
  ('conceal', 'expose'): 0.248
  ('hide', 'reveal'): 0.226

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.226
  ('conceal', 'expose'): 0.248
  ('suppress', 'leak'): 0.249


In [ ]:
# Seems like there is some interaction between the pairs, so I'll remove the weaker ones.
for pair in conj:
    if pair not in [('exposes', 'conceals'), ('revealing', 'hiding')]:
        epistemic_action_axis = epistemic_action_axis.remove_pair(pair)

print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.242

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.315
  ('suppresses', 'leaks'): 0.244
  ('conceal', 'expose'): 0.207

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.203
  ('conceal', 'expose'): 0.207
  ('suppresses', 'leaks'): 0.244


Note that the SemAxis class also allows us to more directly examine the parallelism of antonym pairs.

In [6]:
# Check which pairs fit well and which don't
print("Best pairs (highest parallelism):")
for pair, score in epistemic_action_axis.get_best_pairs(3):
    print(f"  {pair[0]:15s} - {pair[1]:15s} : {score:.3f}")

print("\nWorst pairs (lowest parallelism):")
for pair, score in epistemic_action_axis.get_worst_pairs(3):
    print(f"  {pair[0]:15s} - {pair[1]:15s} : {score:.3f}")

Best pairs (highest parallelism):
  hide            - reveal          : 0.230
  conceal         - expose          : 0.208
  suppress        - leak            : 0.207

Worst pairs (lowest parallelism):
  obfuscate       - illuminate      : 0.171
  suppress        - leak            : 0.207
  conceal         - expose          : 0.208


## Epistemic-action Axis Overview

In [55]:
# Summary of the epistemic-action axis
# Create semantic axis using the SemAxis class defined in semaxis_util.py
epistemic_action_axis = SemAxis(
    [
    ("illuminate", "obfuscate"),
    ('reveal', 'hide'),
    ('leak', 'suppress'),
    ('expose', 'conceal'),
    ('enlighten', 'misrepresent')
], 
    w2v_model, 
    name="reveal_hide"
)

# View axis summary
print(epistemic_action_axis.summary())

Semantic Axis: reveal_hide
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.189

Best pairs (highest parallelism):
  ('illuminate', 'obfuscate'): 0.200
  ('reveal', 'hide'): 0.195
  ('leak', 'suppress'): 0.192

Worst pairs (lowest parallelism):
  ('enlighten', 'misrepresent'): 0.165
  ('expose', 'conceal'): 0.190
  ('leak', 'suppress'): 0.192


## True-False Axis

In [33]:
# Construct new true-false axis; start big then whittle down
true_false_axis = SemAxis(
    [("true", "false"),
    ("authentic", "counterfeit"),
    ("real", "fake"),
    ("accurate", "inaccurate"),
    ("correct", "incorrect"),
    ("factual", "fictitious"),
    ("reliable", "unreliable")], 
    w2v_model, 
    name="true_false"
)
print(true_false_axis.summary())

Semantic Axis: true_false
Number of antonym pairs: 7
Concept vector dimension: 300
Overall parallelism: 0.136

Best pairs (highest parallelism):
  ('accurate', 'inaccurate'): 0.199
  ('true', 'false'): 0.179
  ('real', 'fake'): 0.136

Worst pairs (lowest parallelism):
  ('factual', 'fictitious'): 0.061
  ('authentic', 'counterfeit'): 0.119
  ('correct', 'incorrect'): 0.125


In [34]:
# Remove 2 pairs with lowest parallelism to get to 5
weak = true_false_axis.get_worst_pairs(2)
for pair in weak:
    true_false_axis = true_false_axis.remove_pair(pair[0])
print(true_false_axis.summary())


Semantic Axis: true_false
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.192

Best pairs (highest parallelism):
  ('accurate', 'inaccurate'): 0.220
  ('true', 'false'): 0.204
  ('real', 'fake'): 0.189

Worst pairs (lowest parallelism):
  ('correct', 'incorrect'): 0.170
  ('reliable', 'unreliable'): 0.179
  ('real', 'fake'): 0.189


In [35]:
# Want to add "genuine" to the axis but unsure how to pair it
# Find new pair to add
candidates = ['phony', 'phoney', 'bogus']
best = true_false_axis.find_best_antonym('genuine', candidates)
print(f"Best new pair: genuine-{best[0][0]} ({best[0][1]:.3f})")

Best new pair: genuine-phony (0.216)


In [36]:
# Looking for top N best antonyms
f, r = true_false_axis.find_antonyms_fullsearch("genuine", top_n=20, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: phony - 0.21627798130114873
Word-Score: bogus - 0.2139435147245725
Word-Score: misleading - 0.2099790905912717
Word-Score: spurious - 0.20727827499310175
Word-Score: falsehoods - 0.20551205923159918
Word-Score: fraudulent - 0.20518241276343663
Word-Score: narratives - 0.20496975233157474
Word-Score: fabricated - 0.20430027991533278
Word-Score: flags - 0.20410847514867783
Word-Score: innacurate - 0.20376312385002773
Word-Score: untrustworthy - 0.2032149796684583
Word-Score: flage - 0.20230055501063665
Word-Score: defamatory - 0.2021165614326795
Word-Score: unrelated - 0.20207158774137496
Word-Score: misinfo - 0.2019672855734825
Word-Score: dichotomies - 0.20189115554094314
Word-Score: drumpfs - 0.20153145641088485
Word-Score: unwarranted - 0.20150913347800573
Word-Score: nonsensical - 0.20148483763138453
Word-Score: flawed - 0.20136093348264694


In [37]:
# Adding genuine-bogus and remove real-fake; fits better with the overall theme of validity
true_false_axis = true_false_axis.add_pair(('genuine', 'bogus'))
true_false_axis = true_false_axis.remove_pair(('real', 'fake'))
print(true_false_axis.summary())

Semantic Axis: true_false
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.217

Best pairs (highest parallelism):
  ('genuine', 'bogus'): 0.250
  ('accurate', 'inaccurate'): 0.230
  ('true', 'false'): 0.216

Worst pairs (lowest parallelism):
  ('reliable', 'unreliable'): 0.194
  ('correct', 'incorrect'): 0.194
  ('true', 'false'): 0.216


## True-False Axis Overview

In [ ]:
true_false_axis = SemAxis(
    [("true", "false"),
    ("accurate", "inaccurate"),
    ("correct", "incorrect"),
    ("reliable", "unreliable"),
    ("genuine", "bogus")], 
    w2v_model, 
    name="true_false"
)
print(true_false_axis.summary())

Semantic Axis: true_false
Number of antonym pairs: 6
Concept vector dimension: 300
Overall parallelism: 0.178

Best pairs (highest parallelism):
  ('genuine', 'bogus'): 0.232
  ('true', 'false'): 0.208
  ('accurate', 'inaccurate'): 0.189

Worst pairs (lowest parallelism):
  ('truth', 'illusion'): 0.102
  ('reliable', 'unreliable'): 0.159
  ('correct', 'incorrect'): 0.181


## Good-Evil Axis

Now exploring a good-evil axis. Note that I am using good-evil rather than good-bad to operationalize an explicitly moral evaluation.

In [6]:
good_evil_axis = SemAxis(
    [
        ("good", "evil")
        
    ], 
    w2v_model, 
    name="good_evil"
)
print(good_evil_axis.summary())

Semantic Axis: good_evil
Number of antonym pairs: 1
Concept vector dimension: 300


In [39]:
# Find the 10 closest tokens to the selected token
sim_token = "evil"
most_similar_results = w2v_model.most_similar(sim_token, topn=20)
print(f"Tokens closest to '{sim_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'evil':
wicked               0.604
doers                0.598
satan                0.577
demonic              0.569
demons               0.545
devil                0.524
demon                0.520
evildoers            0.520
luciferians          0.514
malevolent           0.512
winst                0.502
satanic              0.501
wickedness           0.495
deinstallating       0.493
monsters             0.491
psychopaths          0.490
wickness             0.483
soulles              0.477
vile                 0.476
evilness             0.474


In [7]:
# Looking for top N best antonyms
f, r = good_evil_axis.find_antonyms_fullsearch("malevolent", top_n=20, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: decent - 0.42294469475746155
Word-Score: fabulous - 0.41817399859428406
Word-Score: goooooood - 0.4005352258682251
Word-Score: excellent - 0.39703676104545593
Word-Score: wonderful - 0.39141055941581726
Word-Score: cool - 0.38994133472442627
Word-Score: fantastic - 0.3821265399456024
Word-Score: luck - 0.3773020803928375
Word-Score: anywho - 0.37644702196121216
Word-Score: quality - 0.3762781023979187
Word-Score: solid - 0.37602686882019043
Word-Score: helpful - 0.3752482533454895
Word-Score: lovely - 0.3744537830352783
Word-Score: snt - 0.3741408884525299
Word-Score: lightpaper - 0.3716490864753723
Word-Score: niiice - 0.37149345874786377
Word-Score: neegs - 0.370989054441452
Word-Score: worry - 0.3706698715686798
Word-Score: morning - 0.3704174757003784
Word-Score: hey - 0.37019944190979004


In [15]:
# Adding saintly-demonic and righteous-wicked
good_evil_axis = SemAxis(
    [
        ("good", "evil"),
        ("righteous", "wicked")
        
    ], 
    w2v_model, 
    name="good_evil"
)
print(good_evil_axis.summary())

Semantic Axis: good_evil
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.274

Best pairs (highest parallelism):
  ('good', 'evil'): 0.274
  ('righteous', 'wicked'): 0.274

Worst pairs (lowest parallelism):
  ('good', 'evil'): 0.274
  ('righteous', 'wicked'): 0.274


In [16]:
# Checking parallelism of extended antonym pairs
candidates = [("virtuous", "corrupt"),
("pure", "corrupt"),
("noble", "vile"),
("blessed", "accursed"),
("benevolent", "malevolent")]

for good, bad in candidates:
    try:
        best = good_evil_axis.find_best_antonym(good, [bad])
        print(f"Pair parallelism to axis: {good}-{bad} ({best[0][1]:.3f})")
    except ValueError as err:
        print(err)

Pair parallelism to axis: virtuous-corrupt (0.253)
Pair parallelism to axis: pure-corrupt (0.138)
Pair parallelism to axis: noble-vile (0.267)
Pair parallelism to axis: blessed-accursed (0.182)
Pair parallelism to axis: benevolent-malevolent (0.232)


In [18]:
good_evil_axis = SemAxis(
    [
        ("good", "evil"),
        ("righteous", "wicked"),
        ("virtuous", "corrupt"),
        ("noble", "vile"),
        ("benevolent", "malevolent")
    ], 
    w2v_model, 
    name="good_evil"
)
print(good_evil_axis.summary())

Semantic Axis: good_evil
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.223

Best pairs (highest parallelism):
  ('good', 'evil'): 0.250
  ('righteous', 'wicked'): 0.245
  ('noble', 'vile'): 0.232

Worst pairs (lowest parallelism):
  ('benevolent', 'malevolent'): 0.184
  ('virtuous', 'corrupt'): 0.204
  ('noble', 'vile'): 0.232


## Good-Evil Axis Overview

In [19]:
good_evil_axis = SemAxis(
    [
        ("good", "evil"),
        ("righteous", "wicked"),
        ("virtuous", "corrupt"),
        ("noble", "vile"),
        ("benevolent", "malevolent")
    ], 
    w2v_model, 
    name="good_evil"
)
print(good_evil_axis.summary())

Semantic Axis: good_evil
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.223

Best pairs (highest parallelism):
  ('good', 'evil'): 0.250
  ('righteous', 'wicked'): 0.245
  ('noble', 'vile'): 0.232

Worst pairs (lowest parallelism):
  ('benevolent', 'malevolent'): 0.184
  ('virtuous', 'corrupt'): 0.204
  ('noble', 'vile'): 0.232


## Holy-Unholy Axis

In [31]:
holy_unholy_axis = SemAxis(
    [
        ("holy", "unholy"),
        ("divine", "satanic"),
        ("sacred", "profane"),
        ("angelic", "demonic"),
        ("blessed", "cursed")
    ], 
    w2v_model, 
    name="holy_unholy"
)
print(holy_unholy_axis.summary())

Semantic Axis: holy_unholy
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.220

Best pairs (highest parallelism):
  ('divine', 'satanic'): 0.287
  ('angelic', 'demonic'): 0.278
  ('blessed', 'cursed'): 0.184

Worst pairs (lowest parallelism):
  ('sacred', 'profane'): 0.172
  ('holy', 'unholy'): 0.180
  ('blessed', 'cursed'): 0.184


In [25]:
# Find the 10 closest tokens to the selected token
sim_token = "satanic"
most_similar_results = w2v_model.most_similar(sim_token, topn=20)
print(f"Tokens closest to '{sim_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'satanic':
ritual               0.682
rituals              0.680
occult               0.659
luciferian           0.649
illuminati           0.600
demonic              0.595
cult                 0.593
satanist             0.591
santanic             0.586
cults                0.585
masonic              0.574
satanism             0.572
luciferians          0.562
cabal                0.558
satanists            0.556
lucifarian           0.551
worshipping          0.537
worshiping           0.537
worshippers          0.536
elite                0.536


## Holy-Unholy Axis Overview

In [23]:
holy_unholy_axis = SemAxis(
    [
        ("holy", "unholy"),
        ("divine", "satanic"),
        ("sacred", "profane"),
        ("angelic", "demonic"),
        ("blessed", "cursed")
    ], 
    w2v_model, 
    name="holy_unholy"
)
print(holy_unholy_axis.summary())

Semantic Axis: holy_unholy
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.220

Best pairs (highest parallelism):
  ('divine', 'satanic'): 0.287
  ('angelic', 'demonic'): 0.278
  ('blessed', 'cursed'): 0.184

Worst pairs (lowest parallelism):
  ('sacred', 'profane'): 0.172
  ('holy', 'unholy'): 0.180
  ('blessed', 'cursed'): 0.184


## Spiritual-Material Axis

In [30]:
spiritual_material_axis = SemAxis(
    [
        ("spiritual", "materialistic"),
        ("cosmic", "worldly"),
        ("light", "darkness"),
        ("galactic", "earthly")
    ], 
    w2v_model, 
    name="spiritual_material"
)
print(spiritual_material_axis.summary())

Semantic Axis: spiritual_material
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.205

Best pairs (highest parallelism):
  ('cosmic', 'worldly'): 0.280
  ('galactic', 'earthly'): 0.269
  ('light', 'darkness'): 0.143

Worst pairs (lowest parallelism):
  ('spiritual', 'materialistic'): 0.127
  ('light', 'darkness'): 0.143
  ('galactic', 'earthly'): 0.269


In [31]:
# Find the 10 closest tokens to the selected token
sim_token = "ethereal"
most_similar_results = w2v_model.most_similar(sim_token, topn=20)
print(f"Tokens closest to '{sim_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'ethereal':
tapestries           0.494
fairies              0.494
kaleidoscopic        0.481
interwoven           0.472
pastel               0.469
celestial            0.467
etheric              0.466
shimmer              0.464
mystical             0.460
hemi                 0.458
cinematographic      0.457
thunderbeat          0.457
soundscapes          0.455
muqarnas             0.452
translucent          0.451
shimmering           0.450
realm                0.449
morpho               0.448
astral               0.448
orchestral           0.447


In [32]:
# Looking for top N best antonyms
f, r = holy_unholy_axis.find_antonyms_fullsearch("ethereal", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: nasty - 0.2323226769765218
Word-Score: tranny - 0.22936004102230073
Word-Score: devious - 0.2284333328406016
Word-Score: murderous - 0.22805831829706827
Word-Score: caval - 0.22802363336086273
Word-Score: crossdressing - 0.22786981463432313
Word-Score: narcisistic - 0.22780705988407135
Word-Score: traitorous - 0.22735000848770143
Word-Score: degenerate - 0.22715423206488292
Word-Score: hatefilled - 0.22689637144406635
Word-Score: antiamerican - 0.2268551896015803
Word-Score: disgusting - 0.22685380826393764
Word-Score: pedophilic - 0.2268266220887502
Word-Score: luciferian - 0.2267365256945292
Word-Score: pos - 0.22668955028057097
Word-Score: psychopathic - 0.22667090743780136
Word-Score: drunken - 0.22657993535200754
Word-Score: bloodsucking - 0.2262962947289149
Word-Score: demon - 0.22621255243817964
Word-Score: thieving - 0.2259848376115163
Word-Score: vile - 0.22596853971481323
Word-Score: downright - 0.22588987946510314
Word-Score: creepy - 0.22581559121608735
Word-Sco

In [ ]:
# Adding ethereal-fleshly
spiritual_material_axis = SemAxis(
    [   
        ("spiritual", "materialistic"),
        ("cosmic", "worldly"),
        ("light", "darkness"),
        ("galactic", "earthly"),
        ("ethereal", "fleshly")
    ], 
    w2v_model, 
    name="spiritual_material"
)
print(spiritual_material_axis.summary())

Semantic Axis: spiritual_material
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.198

Best pairs (highest parallelism):
  ('cosmic', 'worldly'): 0.290
  ('galactic', 'earthly'): 0.271
  ('ethereal', 'fleshly'): 0.189

Worst pairs (lowest parallelism):
  ('spiritual', 'materialistic'): 0.100
  ('light', 'darkness'): 0.142
  ('ethereal', 'fleshly'): 0.189


In [140]:
# Looking for top N best antonyms
f, r = spiritual_material_axis.find_antonyms_fullsearch("ascension", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: pleasures - 0.25439582467079164
Word-Score: philippians - 0.2533457264304161
Word-Score: materialistic - 0.2525000974535942
Word-Score: sinful - 0.25018865056335926
Word-Score: indulgences - 0.24919258877635003
Word-Score: fleshly - 0.24868930280208587
Word-Score: phillipians - 0.2474123567342758
Word-Score: sin - 0.24640836492180823
Word-Score: forsake - 0.24576284140348434
Word-Score: materialism - 0.2456764444708824
Word-Score: comforts - 0.24408457055687904
Word-Score: munus - 0.24392190873622893
Word-Score: desires - 0.243750611692667
Word-Score: christlikeness - 0.24354711696505546
Word-Score: passions - 0.24339225068688392
Word-Score: stumblingblock - 0.2431899681687355
Word-Score: idolatry - 0.2430393122136593
Word-Score: possessions - 0.24234384782612323
Word-Score: selfishness - 0.24229937493801118
Word-Score: escotology - 0.2421689972281456
Word-Score: moralism - 0.24117792770266533
Word-Score: meak - 0.24094861298799514
Word-Score: gratification - 0.240732371807

In [38]:
# Checking parallelism of extended antonym pairs; asked Claude to help me with suggestions
candidates = [
    ("harmony", "discord"),
    ("unity", "division"),
    ("warriors", "sheeple"),
    ("awakening", "psyops"),
    ("multidimensional", "unidimensional"),
    ("spiritual", "material")
]

for spiritual, mundane in candidates:
    try:
        newax = spiritual_material_axis.add_pair((spiritual, mundane))
        print(f"Parallelism with pair {spiritual}-{mundane}: ({newax.parallelism_score:.3f})")
    except ValueError as err:
        print(err)
   

Parallelism with pair harmony-discord: (0.145)
Parallelism with pair unity-division: (0.148)
Parallelism with pair warriors-sheeple: (0.172)
Parallelism with pair awakening-psyops: (0.151)
Parallelism with pair multidimensional-unidimensional: (0.195)
Parallelism with pair spiritual-material: (0.140)


In [39]:
print(spiritual_material_axis.add_pair(("multidimensional", "unidimensional")).summary())

# Seems like light-darkness is not a great fit, so I'll replace it with multidimension-unidimensional

Semantic Axis: spiritual_material
Number of antonym pairs: 6
Concept vector dimension: 300
Overall parallelism: 0.195

Best pairs (highest parallelism):
  ('cosmic', 'worldly'): 0.291
  ('galactic', 'earthly'): 0.245
  ('multidimensional', 'unidimensional'): 0.188

Worst pairs (lowest parallelism):
  ('spiritual', 'materialistic'): 0.128
  ('light', 'darkness'): 0.139
  ('ethereal', 'fleshly'): 0.179


## Spiritual-Material Axis Overview

In [42]:
spiritual_material_axis = SemAxis(
    [   
        ("spiritual", "materialistic"),
        ("cosmic", "worldly"),
        ("galactic", "earthly"),
        ("ethereal", "fleshly"),
        ("multidimensional", "unidimensional")
    ], 
    w2v_model, 
    name="spiritual_material"
)
print(spiritual_material_axis.summary())

Semantic Axis: spiritual_material
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.223

Best pairs (highest parallelism):
  ('cosmic', 'worldly'): 0.322
  ('galactic', 'earthly'): 0.244
  ('multidimensional', 'unidimensional'): 0.203

Worst pairs (lowest parallelism):
  ('spiritual', 'materialistic'): 0.157
  ('ethereal', 'fleshly'): 0.189
  ('multidimensional', 'unidimensional'): 0.203


## Light-Darkness Axis?

In [175]:
ligth_darkness_axis = SemAxis(
    [
        ("light", "darkness")
    ], 
    w2v_model, 
    name="light_darkness"
)
print(ligth_darkness_axis.summary())

Semantic Axis: light_darkness
Number of antonym pairs: 1
Concept vector dimension: 300


In [178]:
# Looking for top N best antonyms
f, r = ligth_darkness_axis.find_antonyms_fullsearch("illusion", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: 300ghz - 0.23096218705177307
Word-Score: sylvania - 0.22631043195724487
Word-Score: japaese - 0.22332435846328735
Word-Score: leds - 0.21979627013206482
Word-Score: matra - 0.21900679171085358
Word-Score: vinyl - 0.21854563057422638
Word-Score: tement - 0.21839262545108795
Word-Score: muhl - 0.2181476205587387
Word-Score: lig - 0.21754318475723267
Word-Score: biophoton - 0.21645596623420715
Word-Score: biophotons - 0.21238532662391663
Word-Score: tachyonic - 0.20585110783576965
Word-Score: frecuencies - 0.20548683404922485
Word-Score: refracting - 0.20369209349155426
Word-Score: paper - 0.20334970951080322
Word-Score: ultraviolet - 0.2029835730791092
Word-Score: splint - 0.20219632983207703
Word-Score: aper - 0.2020123153924942
Word-Score: hydrodynamics - 0.20089446008205414
Word-Score: laser - 0.19858038425445557
Word-Score: photons - 0.19834987819194794
Word-Score: agirlintheuniverse - 0.1963205635547638
Word-Score: kellert - 0.1962573379278183
Word-Score: photonic - 0.19

## Dictatorship-Democracy Axis

In [ ]:
# Find the 20 nearest neighbors to a word in the w2v_model
word = 'autocracy'
nearest_neighbors = w2v_model.most_similar(word, topn=20)
print(f"20 nearest neighbors to '{word}':")
for word, similarity in nearest_neighbors:
    print(f"{word}: {similarity:.4f}")

20 nearest neighbors to 'autocracy':
democracy: 0.5250
democracies: 0.4664
kratia: 0.4618
dictatorships: 0.4206
unipolarism: 0.4158
socialism: 0.4142
hegemony: 0.4124
plutocracy: 0.4114
authoritarianism: 0.4059
progressivism: 0.4033
keynesianism: 0.4024
dictatorial: 0.3994
somoza: 0.3970
dictatorship: 0.3955
globalism: 0.3915
cosmopolitanism: 0.3910
collectivism: 0.3900
autocracies: 0.3870
autocratic: 0.3865
authoritarian: 0.3861


In [14]:
# Construct new tyranny-resistance axis
political_axis = SemAxis(
    [
    ("dictatorship", "democracy"),
    ("tyranny", "freedom"),
    ("repression", "liberty"),
    ("surveillance", "privacy"),
    ('autocratic', 'constitutional')
    ],
    w2v_model, 
    name="political"
)

In [77]:
# Looking for top N best antonyms
f, r = political_axis.find_antonyms_fullsearch("autocracy", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: totalitarian - 0.31602745873870736
Word-Score: constitutional - 0.26337649954956954
Word-Score: authoritarian - 0.2587964539159985
Word-Score: integrity - 0.24995875074901497
Word-Score: zenmaster - 0.24664277584977534
Word-Score: libertys - 0.23435036793703404
Word-Score: constitution - 0.2335446180629801
Word-Score: worldfreedomalliance - 0.22977886228250316
Word-Score: cherished - 0.22806465179427757
Word-Score: evar - 0.22722582863413032
Word-Score: standupx - 0.2261057755036956
Word-Score: liberties - 0.22418589435368694
Word-Score: upholding - 0.22317502388593177
Word-Score: consitution - 0.220629891850604
Word-Score: mycompass - 0.220012752941995
Word-Score: mutual - 0.21907290260613083
Word-Score: libertate - 0.21901102044987836
Word-Score: cpotus - 0.2176468595390852
Word-Score: technocratic - 0.21731296130740752
Word-Score: sanctity - 0.21670634239878375
Word-Score: heritage - 0.2159451165272801
Word-Score: coxforfreedom - 0.21575378398690442
Word-Score: shuteye -

## Dictatorship-Democracy Axis Overview

In [58]:
political_axis = SemAxis(
    [
    ("democracy", "dictatorship"),
    ("freedom", "tyranny"),
    ("liberty", "repression"),
    ("privacy", "surveillance"),
    ('constitutional', 'autocratic')
    ],
    w2v_model, 
    name="democracy_dictatorship"
)

print(political_axis.summary())

Semantic Axis: democracy_dictatorship
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.230

Best pairs (highest parallelism):
  ('democracy', 'dictatorship'): 0.260
  ('liberty', 'repression'): 0.249
  ('constitutional', 'autocratic'): 0.234

Worst pairs (lowest parallelism):
  ('privacy', 'surveillance'): 0.196
  ('freedom', 'tyranny'): 0.214
  ('constitutional', 'autocratic'): 0.234


## Elites-Population Axis

In [ ]:
# Construct new elite-people axis
elite_axis = SemAxis(
    [
        ('populace', 'elites')
    ],
    w2v_model, 
    name="populace_elites"
)
print(elite_axis.summary())

In [190]:
# Looking for top N best antonyms
f, r = elite_axis.find_antonyms_fullsearch("elites", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: illegitimate - 0.3355028923771709
Word-Score: populations - 0.328298398366036
Word-Score: 550million - 0.2804964984907321
Word-Score: percentageof - 0.2736101494627434
Word-Score: mwuhahaha - 0.2626707475666208
Word-Score: percentages - 0.2591000002701657
Word-Score: depopulate - 0.2556974917305115
Word-Score: manageable - 0.25045213700684854
Word-Score: populace - 0.24537490418052285
Word-Score: poulation - 0.23781413391758172
Word-Score: mortality - 0.2373329449949207
Word-Score: proportion - 0.23526767708590962
Word-Score: culling - 0.23450159485832542
Word-Score: obiden - 0.23441046904792787
Word-Score: 204m - 0.2331161135314565
Word-Score: mollahs - 0.2305510168564952
Word-Score: mullahs - 0.22963605852983465
Word-Score: 500mil - 0.22910618229555513
Word-Score: uptake - 0.22442953550117234
Word-Score: cull - 0.22277686543158914
Word-Score: regimes - 0.22074377778281593
Word-Score: the_great_cull - 0.2207076695445659
Word-Score: 500million - 0.2188145998387544
Word-Scor

In [191]:
elite_axis = elite_axis.add_pair(('elites', 'populace'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.245

Best pairs (highest parallelism):
  ('regime', 'population'): 0.245
  ('elites', 'populace'): 0.245

Worst pairs (lowest parallelism):
  ('regime', 'population'): 0.245
  ('elites', 'populace'): 0.245


In [192]:
# Find the 20 nearest neighbors to a word in the w2v_model
word = 'elites'
nearest_neighbors = w2v_model.most_similar(word, topn=20)
print(f"20 nearest neighbors to '{word}':")
for word, similarity in nearest_neighbors:
    print(f"{word}: {similarity:.4f}")

# Very useful list showing 'dimensions' of elites I could play around;
# glad to see 'deep_state', one of the defined entities, show up here as well

20 nearest neighbors to 'elites':
elite: 0.7399
elitists: 0.5896
politicians: 0.5612
elitist: 0.5572
bankers: 0.5390
billionaires: 0.5304
celebrities: 0.5213
hollywood: 0.5201
cabal: 0.5112
luciferian: 0.5078
technocrats: 0.5002
celebs: 0.4966
sociopaths: 0.4935
psychopaths: 0.4933
satanic: 0.4924
puppets: 0.4898
luciferians: 0.4863
deep_state: 0.4821
illuminati: 0.4813
peasants: 0.4725


In [193]:
# Looking for top N best antonyms
f, r = elite_axis.find_antonyms_fullsearch("billionaires", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: populations - 0.42116660351167123
Word-Score: populous - 0.37632938953179057
Word-Score: mwuhahaha - 0.36900968793673683
Word-Score: km² - 0.3606816599684296
Word-Score: 550million - 0.34902208562226106
Word-Score: percentages - 0.3458405023066814
Word-Score: manageable - 0.34531344790751506
Word-Score: uptake - 0.3389820815110177
Word-Score: proportion - 0.33778567438463725
Word-Score: aegypti - 0.33191225920385536
Word-Score: agriculturally - 0.33182480634095723
Word-Score: 204m - 0.33050964596756194
Word-Score: 500mil - 0.3298485336093096
Word-Score: demoralizes - 0.32876383266528264
Word-Score: habituate - 0.3280435839410941
Word-Score: neuromodelation - 0.32744205676439275
Word-Score: factoring - 0.3271784472823376
Word-Score: intermix - 0.3258461923449469
Word-Score: landmass - 0.32545699400525674
Word-Score: unsuspecting - 0.323962018331664
Word-Score: percentage - 0.32061477964130836
Word-Score: mosquitoes - 0.31890178190381974
Word-Score: census - 0.318438876878240

In [194]:
elite_axis = elite_axis.add_pair(('politicians', 'voters'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.228

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.281
  ('politicians', 'voters'): 0.219
  ('regime', 'population'): 0.183

Worst pairs (lowest parallelism):
  ('regime', 'population'): 0.183
  ('politicians', 'voters'): 0.219
  ('elites', 'populace'): 0.281


In [195]:
# Legitimate versus illegitimate political actors
elite_axis = elite_axis.add_pair(('cabal', 'citizens'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.192

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.261
  ('politicians', 'voters'): 0.195
  ('cabal', 'citizens'): 0.156

Worst pairs (lowest parallelism):
  ('regime', 'population'): 0.155
  ('cabal', 'citizens'): 0.156
  ('politicians', 'voters'): 0.195


In [196]:
# Adding an economic dimension
# Removing regime-population because it seems a bit out of place; that frees 'population' up for a better antonym pair
elite_axis = elite_axis.add_pair(('bankers', 'workers')).remove_pair(("regime", "population"))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.239

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.258
  ('bankers', 'workers'): 0.250
  ('politicians', 'voters'): 0.233

Worst pairs (lowest parallelism):
  ('cabal', 'citizens'): 0.216
  ('politicians', 'voters'): 0.233
  ('bankers', 'workers'): 0.250


In [197]:
# More bio-power-inspired
elite_axis = elite_axis.add_pair(('technocrats', 'population'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.247

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.298
  ('technocrats', 'population'): 0.259
  ('politicians', 'voters'): 0.252

Worst pairs (lowest parallelism):
  ('cabal', 'citizens'): 0.198
  ('bankers', 'workers'): 0.228
  ('politicians', 'voters'): 0.252


In [198]:
# Adding a celeb-'everyman' axis, given the implication of celebs in Adrenochrome theories
elite_axis = elite_axis.add_pair(('celebrities', 'people'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 6
Concept vector dimension: 300
Overall parallelism: 0.247

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.294
  ('politicians', 'voters'): 0.265
  ('technocrats', 'population'): 0.257

Worst pairs (lowest parallelism):
  ('cabal', 'citizens'): 0.207
  ('bankers', 'workers'): 0.210
  ('celebrities', 'people'): 0.247


In [199]:
# Would like to have something like 'Illuminati' in there but not sure how to pair it
f, r = elite_axis.find_antonyms_fullsearch("satanists", top_n=50, refine_pool=500)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: populous - 0.2560258223896935
Word-Score: residents - 0.2554384803488141
Word-Score: adults - 0.2466498762369156
Word-Score: disenfranchising - 0.24639764286222912
Word-Score: americans - 0.24613488430068606
Word-Score: minorities - 0.24586330425171626
Word-Score: inhabitants - 0.24581257289364225
Word-Score: popasnyansky - 0.24515301202024734
Word-Score: ozernoye - 0.2449595509540467
Word-Score: decennial - 0.24447583513600485
Word-Score: georgians - 0.24444219186192467
Word-Score: gagauzia - 0.24432412286599478
Word-Score: gaotang - 0.24403709386076247
Word-Score: eligible - 0.24388672482399715
Word-Score: canadians - 0.24381743726276217
Word-Score: sapporo - 0.24352494520800455
Word-Score: heinsberg - 0.24350366918813615
Word-Score: workforce - 0.2434515640849159
Word-Score: populations - 0.24326359161308833
Word-Score: californians - 0.24294324857848032
Word-Score: 108k - 0.24294082678499676
Word-Score: zaparozhye - 0.2428469466311591
Word-Score: worker - 0.242767806918

In [ ]:
# Experimented a bit with illuminati and luciferiens; the scores aren't bad, but
# I feel like the terms depart a bit from the original one about elites-vs-masses, so not added
print(elite_axis.add_pair(('luciferians', 'christians')).summary())

Semantic Axis: elite-population
Number of antonym pairs: 7
Concept vector dimension: 300
Overall parallelism: 0.243

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.288
  ('politicians', 'voters'): 0.257
  ('technocrats', 'population'): 0.249

Worst pairs (lowest parallelism):
  ('bankers', 'workers'): 0.204
  ('luciferians', 'christians'): 0.232
  ('celebrities', 'people'): 0.232


## Elites-Population Axis Overview

In [ ]:
elite_axis = SemAxis(
    [
        ('populace', 'elites'),
        ('voters', 'politicians'),
        ('population', 'technocrats'),
        ('people', 'celebrities'),
        ('workers', 'bankers'),
        ('citizens', 'cabal')
    ],
    w2v_model, 
    name="populace_elites"
)
print(elite_axis.summary())

Semantic Axis: populace_elites
Number of antonym pairs: 6
Concept vector dimension: 300
Overall parallelism: 0.247

Best pairs (highest parallelism):
  ('populace', 'elites'): 0.294
  ('voters', 'politicians'): 0.265
  ('population', 'technocrats'): 0.257

Worst pairs (lowest parallelism):
  ('citizens', 'cabal'): 0.207
  ('workers', 'bankers'): 0.210
  ('people', 'celebrities'): 0.247


# 3. Comparing Semantic Axes

In [5]:
# Import German and English model and build matched cross-axis definitions
import json
from pathlib import Path

AXES_EN_PATH = Path("/project/ssd-stu-research/ploertscher/thesis_code/ideological_resonance_thesis/w2v/analysis/axes_en.json")
AXES_DE_PATH = Path("/project/ssd-stu-research/ploertscher/thesis_code/ideological_resonance_thesis/w2v/analysis/axes_de.json")
MODEL_EN_NAME = "2_w2v_min10"
MODEL_DE_NAME = "2_w2v_min10_de"

# Reuse the English model already loaded above when available; load German here.
en_kv = w2v_model if "w2v_model" in globals() else helpers.load_trained_w2v_keyed_vectors(MODEL_EN_NAME)
de_kv = helpers.load_trained_w2v_keyed_vectors(MODEL_DE_NAME)

token_to_canonical = helpers.load_w2v_token_to_canonical()

with open(AXES_EN_PATH, "r", encoding="utf-8") as f:
    axes_en = {axis: [tuple(pair) for pair in pairs] for axis, pairs in json.load(f).items()}

with open(AXES_DE_PATH, "r", encoding="utf-8") as f:
    axes_de = {axis: [tuple(pair) for pair in pairs] for axis, pairs in json.load(f).items()}


def unit_vector(vec):
    vec = np.asarray(vec, dtype=np.float64)
    norm = np.linalg.norm(vec)
    if norm <= 0:
        raise ValueError("Cannot normalize a zero vector")
    return vec / norm


# Axes match the 2-D plot from semanalysis_plots.py: x=reveal_hide, y=holy_unholy.
reveal_hide_axis_en = unit_vector(SemAxis(axes_en["reveal_hide"], en_kv, name="reveal_hide").concept_vector)
holy_unholy_axis_en = unit_vector(SemAxis(axes_en["holy_unholy"], en_kv, name="holy_unholy").concept_vector)
reveal_hide_axis_de = unit_vector(SemAxis(axes_de["reveal_hide_de"], de_kv, name="reveal_hide_de").concept_vector)
holy_unholy_axis_de = unit_vector(SemAxis(axes_de["holy_unholy_de"], de_kv, name="holy_unholy_de").concept_vector)

language_axes = {
    "EN": {
        "kv": en_kv,
        "x_axis": reveal_hide_axis_en,
        "y_axis": holy_unholy_axis_en,
        "axis_names": ("reveal_hide", "holy_unholy"),
    },
    "DE": {
        "kv": de_kv,
        "x_axis": reveal_hide_axis_de,
        "y_axis": holy_unholy_axis_de,
        "axis_names": ("reveal_hide_de", "holy_unholy_de"),
    },
}

print("Loaded matched EN/DE models and semantic axes for quadrant target-neighbor analysis.")
print(f"Mapped actor tokens available: {len(token_to_canonical)}")

Loaded matched EN/DE models and semantic axes for quadrant target-neighbor analysis.
Mapped actor tokens available: 132


In [7]:
# Build 300-D target points for each quadrant and inspect their semantic neighborhoods.
# Instead of averaging actors inside a quadrant, anchor each target at the global
# embedding mean and push it to fixed +/-0.5 coordinates on each semantic axis.
TOPN_NEIGHBORS = 50
TARGET_AXIS_OFFSET = 0.5
LANGUAGE_CATEGORY_EXCLUDE_SUFFIXES = {
    "EN": ("_de", "_it"),
    "DE": ("_it", "_us"),
}

quadrants = [
    ("reveal+holy", 1, 1),
    ("reveal+unholy", 1, -1),
    ("hide+holy", -1, 1),
    ("hide+unholy", -1, -1),
]

with open(helpers.ENTITY_MAPPING_2_PATH, "r", encoding="utf-8") as f:
    entity_mapping = json.load(f)

token_to_category = {
    entity_key: category
    for category, entities in entity_mapping.items()
    for entity_key in entities
}


def normalized_vocab_matrix(kv):
    vectors = np.asarray(kv.vectors, dtype=np.float64)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return np.divide(vectors, norms, out=np.zeros_like(vectors), where=norms > 0)


def actor_projection_frame(
    kv,
    x_axis,
    y_axis,
    actor_token_to_canonical,
    token_to_category,
    excluded_category_suffixes=(),
):
    rows = []
    seen_canonicals = set()
    for token, canonical in sorted(actor_token_to_canonical.items(), key=lambda item: item[1]):
        category = token_to_category.get(token)
        if token not in kv or canonical in seen_canonicals:
            continue
        if category is not None and category.endswith(excluded_category_suffixes):
            continue
        vec = unit_vector(kv[token])
        rows.append(
            {
                "token": token,
                "canonical": canonical,
                "category": category,
                "x": float(vec @ x_axis),
                "y": float(vec @ y_axis),
            }
        )
        seen_canonicals.add(canonical)
    return pd.DataFrame(rows)


def quadrant_mask(df, x_sign, y_sign):
    x_ok = df["x"] > 0 if x_sign > 0 else df["x"] < 0
    y_ok = df["y"] > 0 if y_sign > 0 else df["y"] < 0
    return x_ok & y_ok


def neighbor_label(token):
    canonical = token_to_canonical.get(token)
    return f"{token} [{canonical}]" if canonical is not None else token


language_results = {}
summary_rows = []
quadrant_actor_frames = {}

def build_quadrant_targets_and_neighbors(lang, spec, topn=20):
    kv = spec["kv"]
    x_axis = unit_vector(spec["x_axis"])
    y_axis = unit_vector(spec["y_axis"])
    excluded_suffixes = LANGUAGE_CATEGORY_EXCLUDE_SUFFIXES.get(lang, ())
    actor_df = actor_projection_frame(
        kv,
        x_axis,
        y_axis,
        token_to_canonical,
        token_to_category,
        excluded_category_suffixes=excluded_suffixes,
    )
    if actor_df.empty:
        raise ValueError(f"No mapped actors found in {lang} embedding vocabulary.")

    vocab_vectors = normalized_vocab_matrix(kv)
    origin = vocab_vectors.mean(axis=0)

    targets = {
        quadrant_name: origin + x_sign * TARGET_AXIS_OFFSET * x_axis + y_sign * TARGET_AXIS_OFFSET * y_axis
        for quadrant_name, x_sign, y_sign in quadrants
    }
    neighbors = {
        quadrant_name: kv.similar_by_vector(target, topn=topn)
        for quadrant_name, target in targets.items()
    }
    return actor_df, targets, neighbors


for lang, spec in language_axes.items():
    actor_df, targets, neighbors = build_quadrant_targets_and_neighbors(
        lang,
        spec,
        topn=TOPN_NEIGHBORS,
    )
    language_results[lang] = {
        "actor_df": actor_df,
        "targets": targets,
        "neighbors": neighbors,
        "target_offset": TARGET_AXIS_OFFSET,
    }
    quadrant_actor_frames[lang] = actor_df

    excluded_suffixes = LANGUAGE_CATEGORY_EXCLUDE_SUFFIXES.get(lang, ())
    print(f"\n=== {lang}: {spec['axis_names'][0]} x {spec['axis_names'][1]} ===")
    print(f"Actors in embedding vocabulary after language filter: {len(actor_df)}")
    print(f"Excluded actor categories ending with: {excluded_suffixes}")
    print(f"Target quadrant coordinates: +/-{TARGET_AXIS_OFFSET:.2f} on each semantic axis")

    for quadrant_name, x_sign, y_sign in quadrants:
        qdf = actor_df.loc[quadrant_mask(actor_df, x_sign, y_sign)].copy()
        print(f"\n{lang} {quadrant_name}: {len(qdf)} actors in quadrant")
        if not qdf.empty:
            qdf["radius"] = np.sqrt(qdf["x"] ** 2 + qdf["y"] ** 2)
            print("  Most extreme actors in this quadrant's 2-D projection:")
            for row in qdf.sort_values("radius", ascending=False).head(10).itertuples(index=False):
                print(f"    {row.canonical:35s} [{row.category}] x={row.x:+.3f}, y={row.y:+.3f}")

        print(f"  Nearest neighbors to anchored {quadrant_name} target:")
        for rank, (token, score) in enumerate(neighbors[quadrant_name], start=1):
            print(f"    {rank:02d}. {neighbor_label(token):45s} {score:+.3f}")
            summary_rows.append(
                {
                    "language": lang,
                    "quadrant": quadrant_name,
                    "n_actors_in_quadrant": len(qdf),
                    "target_x_coordinate": x_sign * TARGET_AXIS_OFFSET,
                    "target_y_coordinate": y_sign * TARGET_AXIS_OFFSET,
                    "rank": rank,
                    "neighbor": token,
                    "neighbor_label": neighbor_label(token),
                    "similarity": score,
                }
            )

quadrant_target_neighbors = pd.DataFrame(summary_rows)

print("\n=== EN/DE nearest-neighbor comparison by quadrant ===")
for quadrant_name, _x_sign, _y_sign in quadrants:
    print(f"\n── {quadrant_name} ──")
    en_neighbors = language_results["EN"]["neighbors"][quadrant_name]
    de_neighbors = language_results["DE"]["neighbors"][quadrant_name]
    for rank in range(TOPN_NEIGHBORS):
        en_token, en_score = en_neighbors[rank]
        de_token, de_score = de_neighbors[rank]
        print(
            f"{rank + 1:02d}. "
            f"EN {neighbor_label(en_token):38s} {en_score:+.3f}    "
            f"DE {neighbor_label(de_token):38s} {de_score:+.3f}"
        )

display(quadrant_target_neighbors)


=== EN: reveal_hide x holy_unholy ===
Actors in embedding vocabulary after language filter: 60
Excluded actor categories ending with: ('_de', '_it')
Target quadrant coordinates: +/-0.50 on each semantic axis

EN reveal+holy: 16 actors in quadrant
  Most extreme actors in this quadrant's 2-D projection:
    God                                 [spiritual_good] x=+0.120, y=+0.307
    Jesus Christ                        [spiritual_good] x=+0.069, y=+0.286
    Truth Social                        [altmedia_us] x=+0.085, y=+0.110
    Telegram                            [tech] x=+0.086, y=+0.098
    Mike Lindell                        [alt_politics_us] x=+0.058, y=+0.080
    Anons                               [qanon_pro] x=+0.074, y=+0.057
    Patriots                            [qanon_pro] x=+0.039, y=+0.070
    Ghislaine Maxwell                   [globalcabal] x=+0.060, y=+0.029
    Donald Trump                        [donald_trump] x=+0.060, y=+0.029
    Peter McCullough                  

,language,quadrant,n_actors_in_quadrant,target_x_coordinate,target_y_coordinate,rank,neighbor,neighbor_label,similarity
0,EN,reveal+holy,16,0.5,0.5,1,divine,divine,0.495357
1,EN,reveal+holy,16,0.5,0.5,2,gaia,gaia,0.453315
2,EN,reveal+holy,16,0.5,0.5,3,muchlove,muchlove,0.451893
3,EN,reveal+holy,16,0.5,0.5,4,kie,kie,0.448581
4,EN,reveal+holy,16,0.5,0.5,5,blessed,blessed,0.447965
...,...,...,...,...,...,...,...,...,...
395,DE,hide+unholy,15,-0.5,-0.5,46,verfassungsfanaten,verfassungsfanaten,0.394593
396,DE,hide+unholy,15,-0.5,-0.5,47,geschichtsklitterung,geschichtsklitterung,0.394502
397,DE,hide+unholy,15,-0.5,-0.5,48,satanisch,satanisch,0.394439
398,DE,hide+unholy,15,-0.5,-0.5,49,plutoniumbomben,plutoniumbomben,0.393454
